In [ ]:
import time
# Let starter.py compute 12 hours from its own start
global_end_time = 0.0

In [ ]:
!pip uninstall -y tensorflow

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    if not (isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)):
        return False
    if np.any((guess < 0) | (guess > 9)):
        return False
    return True
    
def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    if sorted(permutation) != list(range(10)):
        raise ValueError(f"Invalid permutation descriptor: {descriptor}")
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            # Reject empty or jagged grids
            if not by_rows:
                return None
            row_len = len(by_rows[0])
            if not all(len(r) == row_len for r in by_rows):
                return None
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except Exception as e:
            print(f"[DEBUG] convert_tokens_to_array failed: {type(e).__name__}: {e}")
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))
    @staticmethod
    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not query['train']:
                    break
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        skipped = []
        for k, v in results.items():
            # Donor/source-safe mapping: split only the final test-index suffix.
            base_id, base_nr = k.rsplit('_', 1)
            if base_id not in submission:
                skipped.append(k)
                continue
            target_dict = submission[base_id][int(base_nr)]
            # Preserve baseline semantics for a one-candidate result: duplicate
            # attempt_1 into attempt_2. For zero candidates keep the [[0]]
            # defaults; never index v[-1] on an empty result.
            attempts = list(v[:len(target_dict)])
            if len(attempts) == 1 and len(target_dict) >= 2:
                attempts.append(attempts[0])
            for i, g in enumerate(attempts[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = np.asarray(g).tolist()
        if skipped:
            print(f"WARNING: skipped {len(skipped)} result keys not in submission: {skipped[:10]}")

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np


def hashable(guess):
    return tuple(map(tuple, guess))


def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs


def getter_full_probmul_3(guesses, baseline=None):
    # Dynamic baseline to avoid negative scores (from V1)
    if baseline is None:
        beam_scores = [g["beam_score"] for g in guesses]
        if not beam_scores:
            return 0
        baseline = max(beam_scores) + 1.0
    inf_score = np.sum([baseline - g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline - s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score


def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)


def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score


def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


SELECTOR_CURRENT = "CURRENT"
SELECTOR_IMBUE_EXACT = "IMBUE_EXACT"
SELECTOR_IMBUE_QWEN_DIVERSE = "IMBUE_QWEN_DIVERSE"


class ArcDecoder:

    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    @staticmethod
    def _validate_decoded_sample(sample):
        """Validate and normalize one persisted decoder candidate.

        Required fields:
          - solution: non-empty 2-D ARC grid, integer colors 0..9, max 30x30
          - beam_score: finite scalar
          - score_aug: non-empty 1-D sequence of finite scalars

        Returns:
            (normalized_sample, error_message)
        """
        if not isinstance(sample, dict):
            return None, f"payload is {type(sample).__name__}, expected dict"

        required = ("solution", "beam_score", "score_aug")
        missing = [name for name in required if name not in sample]
        if missing:
            return None, f"missing required field(s): {missing}"

        try:
            solution = np.asarray(sample["solution"])
        except Exception as exc:
            return None, f"solution cannot be converted to ndarray: {exc}"

        if solution.ndim != 2:
            return None, f"solution ndim={solution.ndim}, expected 2"
        if solution.size == 0 or any(int(d) <= 0 for d in solution.shape):
            return None, f"solution has empty/invalid shape {solution.shape}"
        if any(int(d) > 30 for d in solution.shape):
            return None, f"solution shape {solution.shape} exceeds ARC 30x30 bounds"

        try:
            solution_float = solution.astype(np.float64)
        except Exception as exc:
            return None, f"solution is not numeric: {exc}"

        if not np.all(np.isfinite(solution_float)):
            return None, "solution contains NaN/Inf"
        if not np.all(solution_float == np.floor(solution_float)):
            return None, "solution contains non-integer values"
        if np.any(solution_float < 0) or np.any(solution_float > 9):
            return None, "solution contains colors outside 0..9"
        solution = solution_float.astype(np.int64)

        try:
            beam_score = float(sample["beam_score"])
        except Exception as exc:
            return None, f"beam_score is not a scalar number: {exc}"
        if not np.isfinite(beam_score):
            return None, "beam_score is NaN/Inf"

        try:
            score_aug = np.asarray(sample["score_aug"], dtype=np.float64)
        except Exception as exc:
            return None, f"score_aug is not numeric: {exc}"
        if score_aug.ndim != 1:
            return None, f"score_aug ndim={score_aug.ndim}, expected 1"
        if score_aug.size == 0:
            return None, "score_aug is empty"
        if not np.all(np.isfinite(score_aug)):
            return None, "score_aug contains NaN/Inf"

        normalized = dict(sample)
        normalized["solution"] = solution
        normalized["beam_score"] = beam_score
        normalized["score_aug"] = score_aug.tolist()
        return normalized, None

    def load_decoded_results(self, store, run_name=""):
        """Load only well-formed outputs belonging to this exact split dataset.

        Stale/foreign files, corrupt payloads, and malformed samples are skipped
        with explicit warnings so they cannot crash or contaminate selectors.
        """
        self.decoded_results = {}
        expected_keys = set(getattr(self.dataset, "keys", []))
        skipped_foreign = []
        skipped_bad = []

        if not os.path.isdir(store):
            print(f"WARNING: decoded-results directory does not exist: {store}")
            return

        for key in sorted(os.listdir(store)):
            path = os.path.join(store, key)
            if not os.path.isfile(path):
                continue

            base_key = key.split(".")[0]
            if base_key not in expected_keys:
                skipped_foreign.append(key)
                continue

            try:
                with bz2.BZ2File(path) as f:
                    outputs = pickle.load(f)
            except Exception as exc:
                skipped_bad.append((key, f"read error: {exc}"))
                continue

            if not isinstance(outputs, (list, tuple)):
                skipped_bad.append((key, f"unexpected payload type: {type(outputs).__name__}"))
                continue

            valid_outputs = []
            for i, sample in enumerate(outputs):
                normalized, error = self._validate_decoded_sample(sample)
                if error is not None:
                    skipped_bad.append((f"{key}.out{i}", error))
                    continue
                valid_outputs.append((i, normalized))

            if not valid_outputs:
                continue

            self.decoded_results.setdefault(base_key, {})
            for i, sample in valid_outputs:
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

        loaded_keys = set(self.decoded_results)
        missing_expected = sorted(expected_keys - loaded_keys)
        print(
            f"*** Decoded coverage: {len(loaded_keys)}/{len(expected_keys)} expected test outputs "
            f"({len(missing_expected)} missing)"
        )
        if missing_expected:
            print(f"WARNING: missing decoded outputs (first 20): {missing_expected[:20]}")
        if skipped_foreign:
            print(f"WARNING: skipped {len(skipped_foreign)} stale/foreign output files: {skipped_foreign[:10]}")
        if skipped_bad:
            print(f"WARNING: skipped {len(skipped_bad)} invalid output entries: {skipped_bad[:10]}")

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    @staticmethod
    def _first_distinct(candidates, reference):
        """Return the first grid whose exact grid hash differs from reference."""
        if reference is None:
            return candidates[0] if candidates else None
        reference_hash = hashable(reference)
        for cand in candidates:
            if hashable(cand) != reference_hash:
                return cand
        return None

    def run_current_selection(self):
        """Exact compatibility copy of the pre-Imbue notebook's run_dual_selection().

        IMPORTANT: score_sum() already deduplicates grids. Therefore, whenever
        score_full_probmul_3 returns >=2 grids, the first two entries of this
        pool are simply Probmul rank-1 and rank-2; Kgmon cannot affect attempt-2.
        We preserve this method unchanged as the CURRENT control arm.
        """
        results = {}
        for bk, guesses in self.decoded_results.items():
            g = {k: v for k, v in guesses.items()}
            ranked_a = score_full_probmul_3(g)
            ranked_b = score_kgmon(g)

            candidates_pool = []
            seen = set()
            for cand in ranked_a[:3] + ranked_b[:3]:
                h = hashable(cand)
                if h not in seen:
                    seen.add(h)
                    candidates_pool.append(cand)

            top_a = candidates_pool[0] if candidates_pool else None
            top_b = candidates_pool[1] if len(candidates_pool) > 1 else top_a
            results[bk] = [x for x in [top_a, top_b] if x is not None]
        return results

    def run_imbue_exact_selection(self):
        """Portable literal analogue of Imbue's ARC attempt-2 rule.

        Imbue sorts its population by one global fitness score and, independently
        for every challenge/test input, chooses the highest-ranked output that is
        different from attempt-1. In this notebook each test input is already a
        separate base_key (task_id_testIndex), so the closest portable analogue is:
          attempt_1 = Probmul rank-1
          attempt_2 = highest Probmul-ranked exact-grid disagreement.

        Since score_sum() already deduplicates outputs, this normally equals
        Probmul rank-2. This arm exists to verify that fact empirically rather
        than assuming it.
        """
        results = {}
        for bk, guesses in self.decoded_results.items():
            ranked = score_full_probmul_3({k: v for k, v in guesses.items()})
            if not ranked:
                results[bk] = []
                continue
            attempt_1 = ranked[0]
            attempt_2 = self._first_distinct(ranked[1:], attempt_1)
            if attempt_2 is None:
                attempt_2 = attempt_1
            results[bk] = [attempt_1, attempt_2]
        return results

    def run_imbue_qwen_diverse_selection(self):
        """Imbue-inspired heterogeneous-ranker attempt-2 for this Qwen/NVARC decoder.

        Keep the established Probmul rank-1 as the conservative attempt-1.
        For attempt-2, scan the independent Kgmon ranking and take its highest
        exact-grid disagreement with attempt-1. Fall back to the Probmul ranking
        only when Kgmon provides no distinct output.

        This is NOT claimed to be Imbue's literal selector; it ports Imbue's
        highest-quality-disagreement principle to the two existing local rankers.
        """
        results = {}
        for bk, guesses in self.decoded_results.items():
            g = {k: v for k, v in guesses.items()}
            ranked_a = score_full_probmul_3(g)
            ranked_b = score_kgmon(g)
            if not ranked_a:
                results[bk] = []
                continue

            attempt_1 = ranked_a[0]
            attempt_2 = self._first_distinct(ranked_b, attempt_1)
            if attempt_2 is None:
                attempt_2 = self._first_distinct(ranked_a[1:], attempt_1)
            if attempt_2 is None:
                attempt_2 = attempt_1

            results[bk] = [attempt_1, attempt_2]
        return results

    # Backward-compatible name. CURRENT stays the safe control until the A/B
    # benchmark below demonstrates a gain for another selector.
    def run_dual_selection(self):
        return self.run_current_selection()

    @staticmethod
    def available_selectors():
        return (SELECTOR_CURRENT, SELECTOR_IMBUE_EXACT, SELECTOR_IMBUE_QWEN_DIVERSE)

    def run_selector(self, selector_name=SELECTOR_CURRENT):
        name = str(selector_name).strip().upper() or SELECTOR_CURRENT
        methods = {
            SELECTOR_CURRENT: self.run_current_selection,
            SELECTOR_IMBUE_EXACT: self.run_imbue_exact_selection,
            SELECTOR_IMBUE_QWEN_DIVERSE: self.run_imbue_qwen_diverse_selection,
        }
        if name not in methods:
            raise ValueError(f"Unknown selector '{selector_name}'. Choose one of {tuple(methods)}")
        return methods[name]()

    def benchmark_selection_algos(self):
        """Benchmark rankers/selectors on *all* expected labelled Eval outputs.

        Missing decoded outputs count as misses.  Foreign decoded keys are ignored.
        This prevents an incomplete inference run from inflating pass@2 accuracy.
        """
        print("*** Benchmark selection algorithms + pass@2 selectors...")

        expected_keys = [
            k for k in getattr(self.dataset, "keys", [])
            if k in getattr(self.dataset, "replies", {})
        ]
        if not expected_keys:
            raise ValueError("benchmark_selection_algos requires labelled dataset.replies")

        labels = {k: self.dataset.replies[k][0] for k in expected_keys}
        expected_set = set(expected_keys)

        num_tasks_per_puzzle = {}
        for basekey in expected_keys:
            puzzle_id, _ = basekey.rsplit("_", 1)
            num_tasks_per_puzzle[puzzle_id] = num_tasks_per_puzzle.get(puzzle_id, 0) + 1

        num_puzzles = len(num_tasks_per_puzzle)
        decoded_expected = expected_set & set(self.decoded_results)
        missing_expected = sorted(expected_set - decoded_expected)
        foreign_decoded = sorted(set(self.decoded_results) - expected_set)

        print(
            f" expected labelled outputs: {len(expected_keys)} | "
            f"decoded: {len(decoded_expected)} | missing: {len(missing_expected)} | "
            f"puzzles: {num_puzzles}"
        )
        if missing_expected:
            print(f" missing outputs count as misses (first 20): {missing_expected[:20]}")
        if foreign_decoded:
            print(f" WARNING: ignoring {len(foreign_decoded)} foreign decoded keys: {foreign_decoded[:10]}")

        num_solved_candidates = 0
        num_total_candidates = 0
        correct_beam_scores = []

        # Candidate-level diagnostics only over available expected outputs.
        for basekey in expected_keys:
            correct_solution = labels[basekey]
            basevalues = self.decoded_results.get(basekey, {})
            for subkey, sample in basevalues.items():
                solution = sample["solution"]
                beam_score = sample.get("beam_score", np.nan)
                score_aug = sample.get("score_aug", [])
                aug_mean = np.mean(score_aug) if len(score_aug) else np.nan

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_candidates += 1
                    if np.isfinite(beam_score):
                        correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"
                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_candidates += 1

        print(f" candidate hits: {num_solved_candidates}/{num_total_candidates}")
        if correct_beam_scores:
            print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
            print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")
        else:
            print(" avg correct beam score: N/A (no correct solutions found)")
            print(" max correct beam score: N/A (no correct solutions found)")

        def score_selected(selected):
            # Iterate over ALL expected labels, not only decoded_results.
            correct_keys = set()
            for k, correct_solution in labels.items():
                attempts = selected.get(k, [])[:self.n_guesses]
                if any(np.array_equal(guess, correct_solution) for guess in attempts):
                    correct_keys.add(k)

            score = sum(
                1.0 / num_tasks_per_puzzle[k.rsplit("_", 1)[0]]
                for k in correct_keys
            )
            return score, correct_keys

        # Original single-ranker top-2 baselines.
        ranker_metrics = {}
        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            score, correct_keys = score_selected(selected)
            ranker_metrics[name] = {"score": score, "correct_keys": correct_keys}
            print(f" acc: {score:5.2f}/{num_puzzles:3} ('{name}') hits={len(correct_keys)} outputs")

        # Selector A/B/C benchmark. CURRENT and IMBUE_EXACT are kept as a
        # control/sanity pair even though they are structurally equivalent here.
        selector_metrics = {}
        selected_cache = {}
        print("\n*** Selector A/B/C benchmark (two attempts each) ***")
        for name in self.available_selectors():
            selected = self.run_selector(name)
            selected_cache[name] = selected
            score, correct_keys = score_selected(selected)
            selector_metrics[name] = {"score": score, "correct_keys": correct_keys}
            print(f" {name:24s}: {score:5.2f}/{num_puzzles:3}  hits={len(correct_keys)} outputs")

        current_score = selector_metrics[SELECTOR_CURRENT]["score"]
        for name in (SELECTOR_IMBUE_EXACT, SELECTOR_IMBUE_QWEN_DIVERSE):
            delta = selector_metrics[name]["score"] - current_score
            print(f" delta {name:19s} vs CURRENT: {delta:+.4f}")

        current_selected = selected_cache[SELECTOR_CURRENT]
        exact_selected = selected_cache[SELECTOR_IMBUE_EXACT]
        comparable_keys = sorted(expected_set)
        current_equals_exact = all(
            len(current_selected.get(k, [])) == len(exact_selected.get(k, []))
            and all(
                np.array_equal(a, b)
                for a, b in zip(current_selected.get(k, []), exact_selected.get(k, []))
            )
            for k in comparable_keys
        )
        print(f" CURRENT == IMBUE_EXACT on this candidate pool: {current_equals_exact}")

        best_selector = max(
            self.available_selectors(),
            key=lambda name: selector_metrics[name]["score"],
        )
        print(f" best selector on this labelled run: {best_selector}")
        print(" NOTE: production selector is NOT auto-switched; keep CURRENT until repeated validation supports a change.")

        return {
            "num_puzzles": num_puzzles,
            "num_expected_outputs": len(expected_keys),
            "num_decoded_outputs": len(decoded_expected),
            "missing_outputs": missing_expected,
            "foreign_decoded_keys": foreign_decoded,
            "rankers": ranker_metrics,
            "selectors": selector_metrics,
            "current_equals_imbue_exact": current_equals_exact,
            "best_selector": best_selector,
        }


In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import copy
import queue
import torch
import numpy as np
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

import hashlib

def deterministic_seed(s, mod=10000):
    """
    Python's built-in hash() is randomized by PYTHONHASHSEED,
    so it gives different values across processes/runs.
    This function provides a consistent, reproducible seed across all runs.
    """
    return int(hashlib.md5(s.encode()).hexdigest(), 16) % mod

logging.disable(logging.WARNING)

# ------------------------------------------------------------
# Fixed token IDs (as used in fine-tuning)
# ------------------------------------------------------------
ARC_VOCAB = {
    "0": 0, "1": 1, "2": 2, "3": 3, "4": 4,
    "5": 5, "6": 6, "7": 7, "8": 8, "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15

MAX_BEAM_CANDIDATES = 5
class UnslothFixedTrainer(UnslothTrainer):
    # Issue https://github.com/unslothai/unsloth/issues/2435
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        if hasattr(loss, "clone"):
            loss = loss.clone()
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            # Ensure labels is 1D (avoid scalar tensor)
            if labels.dim() == 0:
                labels = labels.unsqueeze(0)
            # Use torch.nonzero instead of torch.where to avoid bool scalar issues
            user_mask = (labels == USER_TOKEN_ID)
            assistant_mask = (labels == ASSISTANT_TOKEN_ID)
            user_start_idx = torch.nonzero(user_mask, as_tuple=True)[0].tolist()
            assistant_start_idx = torch.nonzero(assistant_mask, as_tuple=True)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            eos_mask = (labels == EOS_ID)
            end_idx = torch.nonzero(eos_mask, as_tuple=True)[0].tolist()
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                if start >= end:
                    continue
                if j % 2 == 1:  # assistant response part
                    start += 2
                    end += 1
                    if start < end and end <= labels.size(0):
                        batch["labels"][i, start:end] = labels[start:end]
        return batch


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time):
    n = logits.size(0)
    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)
    suffixes = defaultdict(list)
    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i].sort(key=lambda x: x[0])
        candidates[i] = candidates[i][:MAX_BEAM_CANDIDATES]

    while time.time() - start_time < 540 and time.time() < end_time:
        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000.0)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        # Direct cache pass (fast, no clone)
        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_cache = outputs.past_key_values

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens - 1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos + 1,
            cache=next_cache,
            start_time=start_time,
            end_time=end_time,
        )

        del outputs
        # torch.cuda.empty_cache() removed for speed

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x: x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    
    max_len = max(batch_lengths)
    padded_tokens = []
    attention_masks = []
    for tokens in batch_tokens:
        pad_len = max_len - len(tokens)
        padded = tokens + [PAD_ID] * pad_len
        padded_tokens.append(padded)
        attention_masks.append([1] * len(tokens) + [0] * pad_len)
        
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    attention_mask = torch.tensor(attention_masks, device=model.device, dtype=torch.long)
    
    # FIX: use_cache=False because padding breaks KV cache
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True, use_cache=False)
    
    result = []
    # FIX: Slicing on GPU before transferring to CPU to prevent massive OOM Crash
    for i, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_logits = outputs.logits[i, query_length-1 : query_length-1+len(answer_tokens), :]
        answer_log_probs = answer_logits.float().log_softmax(-1).cpu()
        answer_score = answer_log_probs[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
        
    return result


def worker(rank, p_queue, end_time):
    # CUDA optimizations (from winning solution)
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(False)

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch_fused",  # Fused optimizer for speed
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
        dtype=torch.bfloat16,            # Force bfloat16
        attn_implementation="sdpa",      # Use SDPA (fast, memory efficient)
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(tokenizer=tokenizer, mlm=False)

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()
    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    # Fixed queue race condition with Timeout
    while True:
        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        try:
            key = p_queue.get(timeout=60)
        except queue.Empty:
            continue
            
        if key is None:
            break

        start_time = time.time()
        torch.cuda.reset_peak_memory_stats()

        set_peft_model_state_dict(model, default_weights.copy(), adapter_name="default")
        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        # Puzzle-specific seed for diversity (deterministic across runs/processes)
        train_seed = deterministic_seed(key)
        print(f"[Rank {rank}] train_seed={train_seed} for puzzle {key}")
        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=train_seed)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )
            stats = trainer.train()
            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)
            del trainer

        model = FastLanguageModel.for_inference(model)

        gc.collect()
        torch.cuda.empty_cache()

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")
        torch.cuda.reset_peak_memory_stats()
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        # FIX: Use puzzle-specific, deterministic seed for evaluation augmentation
        eval_seed = deterministic_seed(f"{key}_eval")
        print(f"[Rank {rank}] eval_seed={eval_seed} for puzzle {key}")
        eval_ds = puzzle_ds_multi.augment(n=2, seed=eval_seed)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length - max_new_tokens)
   
        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            if batch:   # <-- Empty batch guard
                batches.append(batch)
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            if batch:   # <-- Empty batch guard
                batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            if batch:   # <-- Empty batch guard
                batches.append(batch)
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            if batch:   # <-- Empty batch guard
                batches.append(batch)

        with torch.inference_mode():
            known_scores = {}

            for subkeys in batches:
                spend_time = time.time() - start_time
                if spend_time > 1600 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(
                    model, tokens, max_new_tokens, max_score, end_time
                )

                for subkey_id, scored_beams in dfs_result:
                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:
                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)
                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_seed = deterministic_seed(f"{bk}|aug", mod=1024**2)
                            aug_dataset = aug_dataset.augment(seed=aug_seed)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length - max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores

                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if decoded_result:
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)
                    else:
                        # Do not silently treat an attempted query as if it never ran.
                        # The downstream benchmark will count this missing output as a miss.
                        print(f"[Rank {rank}] WARNING: no valid decoded grid for {subkey}")

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import shutil
import numpy as np
import torch.multiprocessing as mp

# ──────────────────────────────────────────────────────────────
#   Debug mode: only these puzzles run during manual execution.
#   In Kaggle rerun mode, all puzzles are processed automatically.
#   All IDs are from the EVALUATION set (arc-agi_evaluation_challenges.json)
#   to ensure they are found during manual runs.
# ──────────────────────────────────────────────────────────────

DEBUG_PUZZLES = [
    # ── Core 4 (already present) ──
    "0934a4d8",
    "36a08778",
    "981571dc",
    "aa4ec2a5",

    # # ── New additions to test our fixes ──
    # "135a2760",      # Small grid (29x29), simple pattern → tests basic inference pipeline
    # "b5ca7ac4",      # Medium size (22x22), dynamic shape → tests dynamic baseline & dual selection
    # "7b0280bc",      # Large grid (30x30) → tests memory handling & empty batch guard
    # "8b9c3697",      # Dynamic shape change (23x27) → tests invert_mod & jagged row rejection
    # "a395ee82",      # Medium difficulty → tests eval_seed separation effect
    # "65b59efc",      # Large grid + dynamic → tests 1450s timeout & multi-batch processing
    # "7666fa5d",      # Heavy color permutation → tests permutation augmentation & score_aug diversity
    # "1818057f",      # Multiple possible correct solutions → tests dual selection (top_a vs top_b)
    # "2d0172a1",      # Near timeout risk → tests the 1450-second timeout logic
    # "3e6067c3",      # Dynamic + 2 training examples → tests eval_seed vs train_seed separation
    # "6e4f6532",      # Rare color (magenta/maroon) → tests color permutation augmentation

    # # ── NEW: Large + Dynamic (extreme ratio / timeout risk) ──
    # "20a9e565",      # Very large (880 cells), dynamic, ratio 50.0 → tests extreme scale & timeout
    # "4c7dc4dd",      # Max size (900 cells), dynamic, ratio 56.25 → tests memory & jagged row handling
    # "89565ca0",      # Large (557 cells), dynamic, ratio 51.33 → tests high ratio & score_aug diversity
    # "7b3084d4",      # Large (400 cells), dynamic, ratio 25.0 → tests invert_mod & multi-batch
    # "fc7cae8d",      # Large (484 cells), dynamic, ratio 19.36 → tests extreme ratio & empty batch guard

    # # ── NEW: Multi-query puzzles (batch processing & dual selection) ──
    # "1ae2feb7",      # 3 test queries → tests batch processing & dual selection on multi-query
    # "b6f77b65",      # 3 test queries → tests batch processing with multiple candidates
    # "13e47133",      # 2 test queries + rare colors (10 colors) → tests dual selection & color aug

    # # ── NEW: Rare Color + Multi-query (8 picks from EDA) ──
    # "195c6913",      # Size 550, Ratio 1.0, Rare, 2 queries
    # "62593bfd",      # Size 716, Ratio 1.0, Rare, 2 queries (largest)
    # "4e34c42c",      # Size 588, Ratio 9.58, Rare, 2 queries (scaling)
    # "a32d8b75",      # Size 600, Ratio 1.25, Rare, 2 queries
    # "38007db0",      # Size 418, Ratio 3.57, Rare, 2 queries
    # "3a25b0d8",      # Size 484, Ratio 5.32, Rare, 2 queries
    # "5dbc8537",      # Size 312, Ratio 2.22, Rare, 2 queries
    # "d8e07eb2",      # Size 660, Ratio 1.0, Rare, 2 queries

    # # ── NEW: Hardest Tier 1 (4 picks from EDA) ──
    # "6ffbe589",      # Size 400, Ratio 4.0, 8 colors, Dynamic=True → tests dynamic & scaling in Tier 1
    # "58f5dbd5",      # Size 386, Ratio 2.71, 9 colors, Dynamic=True → tests moderate scaling & color diversity
    # "5545f144",      # Size 259, Ratio 4.50, 7 colors, Dynamic=True → tests higher ratio (4.5) in Tier 1
    # "291dc1e1",      # Size 164, Ratio 2.75, 9 colors, Dynamic=True → tests dynamic with many colors
]


def get_puzzle_priority(key, data):
    train = data[key]['train']
    n_train = len(train)

    avg_input_size = np.mean([
        np.prod(np.shape(x['input'])) for x in train
    ])
    size_ratios = [
        max(np.prod(np.shape(x['output'])),
            np.prod(np.shape(x['input']))) /
        max(min(np.prod(np.shape(x['output'])),
                np.prod(np.shape(x['input']))), 1)
        for x in train
    ]
    max_ratio = max(size_ratios)
    avg_ratio = np.mean(size_ratios)

    n_colors = len(set(
        int(c)
        for x in train
        for arr in [x['input'], x['output']]
        for c in np.unique(arr).tolist()
    ))
    is_dynamic = any(
        np.shape(x['input']) != np.shape(x['output'])
        for x in train
    )

    if is_dynamic and max_ratio > 30:
        tier = 3
    elif is_dynamic and max_ratio > 5:
        tier = 2
    elif avg_input_size > 400:
        tier = 2
    else:
        tier = 1

    fine = (
        avg_input_size / 900 * 0.30 +
        (avg_ratio - 1) / 10 * 0.25 +
        n_colors / 10 * 0.20 +
        (0.25 if is_dynamic else 0.0)
    )

    if n_train >= 6 and tier < 3:
        fine *= 0.7

    return (tier, fine)


def local_worker(rank, queue, end_time):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    os.environ["UNSLOTH_DISABLE_RL_PATCH"] = "1"
    os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"

    # Wait for previous worker's ready file with a 300-second timeout
    if rank > 0:
        start_wait = time.time()
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)
            if time.time() - start_wait > 300:
                raise TimeoutError(
                    f"Worker {rank}: Timeout waiting for worker {rank-1} to create sync file."
                )

    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")

    print(f"[Rank {rank}] start!")
    worker(rank, queue, end_time)
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0,
                        help="Unix timestamp when execution must stop. If 0, compute 12 hours from now.")
    args = parser.parse_args()

    if args.end_time == 0.0:
        end_time = time.time() + 12 * 3600
    else:
        end_time = args.end_time

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    # Start every run from a clean output/synchronization state.  This prevents
    # stale files from an earlier execution in the same Kaggle session from
    # contaminating selector ranking or causing workers to skip synchronization.
    output_dir = "/kaggle/inference_outputs"
    if os.path.isdir(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)
    for worker_rank in range(4):
        sync_file = f"/kaggle/worker{worker_rank}"
        try:
            os.remove(sync_file)
        except FileNotFoundError:
            pass

    queue = mp.Manager().Queue()

    # ── EDA-based puzzle ordering ────────────────────────────
    keys_sorted = sorted(
        data.keys(),
        key=lambda k: get_puzzle_priority(k, data)
    )

    # Log tier breakdown
    tiers = [get_puzzle_priority(k, data)[0] for k in keys_sorted]
    print(f"[Starter] Puzzle tiers — "
          f"Easy: {tiers.count(1)}, "
          f"Hard: {tiers.count(2)}, "
          f"Extreme: {tiers.count(3)}")
    print(f"[Starter] First: {keys_sorted[0]}, "
          f"Last: {keys_sorted[-1]}")

    # In rerun mode: all puzzles (ordered). Manual mode: only DEBUG_PUZZLES.
    for key in keys_sorted:
        if not rerun_mode:
            if key not in DEBUG_PUZZLES:
                continue
        queue.put(key)

    # 4 sentinel values – one for each worker
    for _ in range(4):
        queue.put(None)

    mp.spawn(local_worker, args=(queue, end_time), nprocs=4)

In [ ]:
!UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python starter.py --end-time {global_end_time}

In [ ]:
import os
import json
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

if rerun_mode:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")
else:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
    data = data.load_replies("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json")

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
decoder.load_decoded_results("/kaggle/inference_outputs")

# Missing expected outputs remain absent here and are scored as misses by the fixed benchmark.

# SAFE DEFAULT: preserve the previous LB submission behavior until the labelled
# Eval A/B/C benchmark shows that an alternative is better and stable.
# To test a different selector without editing this cell, set ARC_SELECTOR_MODE
# to IMBUE_EXACT or IMBUE_QWEN_DIVERSE before running it.
PRODUCTION_SELECTOR = os.getenv("ARC_SELECTOR_MODE", "CURRENT").strip().upper() or "CURRENT"
print(f"*** Active production selector: {PRODUCTION_SELECTOR}")

selected_results = decoder.run_selector(PRODUCTION_SELECTOR)
submission = data.get_submission(selected_results)

with open("submission.json", "w") as f:
    json.dump(submission, f)

if not rerun_mode:
    selector_benchmark = decoder.benchmark_selection_algos()
    with open("submission.json", "r") as f:
        reload_submission = json.load(f)
    print(f"*** Reload score ({PRODUCTION_SELECTOR}):", data.validate_submission(reload_submission))

    current_score = selector_benchmark["selectors"]["CURRENT"]["score"]
    diverse_score = selector_benchmark["selectors"]["IMBUE_QWEN_DIVERSE"]["score"]
    if diverse_score > current_score:
        print("*** IMBUE_QWEN_DIVERSE improved this labelled run. Re-run/confirm before changing production default.")
    elif diverse_score == current_score:
        print("*** IMBUE_QWEN_DIVERSE tied CURRENT on this labelled run; no evidence yet to switch production.")
    else:
        print("*** IMBUE_QWEN_DIVERSE regressed on this labelled run; keep CURRENT.")


In [ ]:
import numpy as np

if decoder.decoded_results:
    test_key = list(decoder.decoded_results.keys())[0]
    print(f"Key: {test_key}")
    for selector_name in decoder.available_selectors():
        result = decoder.run_selector(selector_name)
        attempts = result.get(test_key, [])
        a1_shape = attempts[0].shape if len(attempts) > 0 else None
        a2_shape = attempts[1].shape if len(attempts) > 1 else None
        distinct = (
            len(attempts) > 1
            and not np.array_equal(attempts[0], attempts[1])
        )
        print(f"{selector_name:24s} attempt_1={a1_shape} attempt_2={a2_shape} distinct={distinct}")
else:
    print("No results were generated. Model inference likely failed.")


In [ ]:
# ================================================================
# DEBUG PUZZLES — COMPREHENSIVE DIAGNOSTIC SUMMARY (FIXED)
# ================================================================
from arc_solver import deterministic_seed
from starter import get_puzzle_priority
from arc_decoder import score_full_probmul_3, score_kgmon
import re
import numpy as np

# ── Ensure data is a dictionary ──
if hasattr(data, 'queries'):
    data_dict = data.queries   # ArcDataset object
else:
    data_dict = data           # Already a dict

DEBUG_PUZZLES = [
    # ── Core 4 (already present) ──
    "0934a4d8",
    "36a08778",
    "981571dc",
    "aa4ec2a5",

    # # ── New additions to test our fixes ──
    # "135a2760",      # Small grid (29x29), simple pattern → tests basic inference pipeline
    # "b5ca7ac4",      # Medium size (22x22), dynamic shape → tests dynamic baseline & dual selection
    # "7b0280bc",      # Large grid (30x30) → tests memory handling & empty batch guard
    # "8b9c3697",      # Dynamic shape change (23x27) → tests invert_mod & jagged row rejection
    # "a395ee82",      # Medium difficulty → tests eval_seed separation effect
    # "65b59efc",      # Large grid + dynamic → tests 1450s timeout & multi-batch processing
    # "7666fa5d",      # Heavy color permutation → tests permutation augmentation & score_aug diversity
    # "1818057f",      # Multiple possible correct solutions → tests dual selection (top_a vs top_b)
    # "2d0172a1",      # Near timeout risk → tests the 1450-second timeout logic
    # "3e6067c3",      # Dynamic + 2 training examples → tests eval_seed vs train_seed separation
    # "6e4f6532",      # Rare color (magenta/maroon) → tests color permutation augmentation

    # # ── NEW: Large + Dynamic (extreme ratio / timeout risk) ──
    # "20a9e565",      # Very large (880 cells), dynamic, ratio 50.0 → tests extreme scale & timeout
    # "4c7dc4dd",      # Max size (900 cells), dynamic, ratio 56.25 → tests memory & jagged row handling
    # "89565ca0",      # Large (557 cells), dynamic, ratio 51.33 → tests high ratio & score_aug diversity
    # "7b3084d4",      # Large (400 cells), dynamic, ratio 25.0 → tests invert_mod & multi-batch
    # "fc7cae8d",      # Large (484 cells), dynamic, ratio 19.36 → tests extreme ratio & empty batch guard

    # # ── NEW: Multi-query puzzles (batch processing & dual selection) ──
    # "1ae2feb7",      # 3 test queries → tests batch processing & dual selection on multi-query
    # "b6f77b65",      # 3 test queries → tests batch processing with multiple candidates
    # "13e47133",      # 2 test queries + rare colors (10 colors) → tests dual selection & color aug

    # # ── NEW: Rare Color + Multi-query (8 picks from EDA) ──
    # "195c6913",      # Size 550, Ratio 1.0, Rare, 2 queries
    # "62593bfd",      # Size 716, Ratio 1.0, Rare, 2 queries (largest)
    # "4e34c42c",      # Size 588, Ratio 9.58, Rare, 2 queries (scaling)
    # "a32d8b75",      # Size 600, Ratio 1.25, Rare, 2 queries
    # "38007db0",      # Size 418, Ratio 3.57, Rare, 2 queries
    # "3a25b0d8",      # Size 484, Ratio 5.32, Rare, 2 queries
    # "5dbc8537",      # Size 312, Ratio 2.22, Rare, 2 queries
    # "d8e07eb2",      # Size 660, Ratio 1.0, Rare, 2 queries

    # # ── NEW: Hardest Tier 1 (4 picks from EDA) ──
    # "6ffbe589",      # Size 400, Ratio 4.0, 8 colors, Dynamic=True → tests dynamic & scaling in Tier 1
    # "58f5dbd5",      # Size 386, Ratio 2.71, 9 colors, Dynamic=True → tests moderate scaling & color diversity
    # "5545f144",      # Size 259, Ratio 4.50, 7 colors, Dynamic=True → tests higher ratio (4.5) in Tier 1
    # "291dc1e1",      # Size 164, Ratio 2.75, 9 colors, Dynamic=True → tests dynamic with many colors
]

# ----------------------------------------------------------------
# CHECK 1 — SEED DETERMINISM
# (Confirms whether deterministic_seed() actually fixed the
#  PYTHONHASHSEED randomization bug from hash())
# ----------------------------------------------------------------
print("=" * 100)
print("CHECK 1: SEED DETERMINISM")
print("=" * 100)
for k in DEBUG_PUZZLES: 
    ts = deterministic_seed(k)
    es = deterministic_seed(f"{k}_eval")
    print(f"{k}: train_seed={ts}, eval_seed={es}")
print("(These values must stay IDENTICAL across every run/commit.)")


# ----------------------------------------------------------------
# CHECK 2 — PUZZLE ORDERING / TIER ASSIGNMENT
# (Confirms get_puzzle_priority() is tiering puzzles sensibly)
# ----------------------------------------------------------------
print("\n" + "=" * 100)
print("CHECK 2: PUZZLE ORDERING (TIER) FOR DEBUG_PUZZLES")
print("=" * 100)
tier_counts = {1: 0, 2: 0, 3: 0}
for k in DEBUG_PUZZLES:
    if k in data_dict:
        tier, fine = get_puzzle_priority(k, data_dict)
        tier_counts[tier] = tier_counts.get(tier, 0) + 1
        print(f"{k}: tier={tier}, fine_score={fine:.4f}")
    else:
        print(f"{k}: NOT FOUND in current dataset")

print(f"\nTier distribution: Easy (1): {tier_counts.get(1, 0)}, "
      f"Hard (2): {tier_counts.get(2, 0)}, "
      f"Extreme (3): {tier_counts.get(3, 0)}")


# ----------------------------------------------------------------
# CHECK 3 — SEED SEPARATION VERIFICATION
# (Confirms train_seed and eval_seed are different for ALL puzzles)
# ----------------------------------------------------------------
print("\n" + "=" * 100)
print("CHECK 3: TRAIN/EVAL SEED SEPARATION")
print("=" * 100)
all_separated = True
for k in DEBUG_PUZZLES:
    ts = deterministic_seed(k)
    es = deterministic_seed(f"{k}_eval")
    if ts == es:
        all_separated = False
        print(f"❌ {k}: train_seed={ts} == eval_seed={es} (NOT SEPARATED!)")
    else:
        print(f"✅ {k}: train_seed={ts} != eval_seed={es}")

if all_separated:
    print("\n✅ All train/eval seeds are properly separated.")
else:
    print("\n❌ Some puzzles have identical train/eval seeds.")


# ----------------------------------------------------------------
# 4. MAIN SOLVE/UNSOLVE + SELECTOR A/B/C SUMMARY
# ----------------------------------------------------------------
print("\n" + "=" * 118)
print("CHECK 4: CANDIDATE COVERAGE + ACTUAL TWO-ATTEMPT SELECTOR HITS")
print("=" * 118)

SELECTOR_NAMES = ["CURRENT", "IMBUE_EXACT", "IMBUE_QWEN_DIVERSE"]
selector_outputs = {name: decoder.run_selector(name) for name in SELECTOR_NAMES}
summary_data = []

for puzzle_id in DEBUG_PUZZLES:
    expected_queries = len(data_dict.get(puzzle_id, {}).get("test", [])) if puzzle_id in data_dict else 0
    if expected_queries <= 0:
        expected_queries = 1
    expected_keys = [f"{puzzle_id}_{i}" for i in range(expected_queries)]

    observed_queries = 0
    candidate_hits = 0
    selector_hits = {name: 0 for name in SELECTOR_NAMES}
    rank_probmul = []
    rank_kgmon = []
    best_beam = None

    for bk in expected_keys:
        guesses = decoder.decoded_results.get(bk, {})
        if not guesses:
            rank_probmul.append(None)
            rank_kgmon.append(None)
            continue
        observed_queries += 1

        correct_solution = decoder.dataset.replies.get(bk, [None])[0]
        if correct_solution is None:
            rank_probmul.append(None)
            rank_kgmon.append(None)
            continue

        correct_samples = [
            g for g in guesses.values()
            if np.array_equal(g["solution"], correct_solution)
        ]
        if correct_samples:
            candidate_hits += 1
            local_best_beam = min(g["beam_score"] for g in correct_samples)
            if best_beam is None or local_best_beam < best_beam:
                best_beam = local_best_beam

        ranked_a = score_full_probmul_3(guesses)
        ranked_b = score_kgmon(guesses)

        def correct_rank(ranked):
            for i, cand in enumerate(ranked, start=1):
                if np.array_equal(cand, correct_solution):
                    return i
            return None

        rank_probmul.append(correct_rank(ranked_a))
        rank_kgmon.append(correct_rank(ranked_b))

        for name in SELECTOR_NAMES:
            attempts = selector_outputs[name].get(bk, [])[:2]
            if any(np.array_equal(cand, correct_solution) for cand in attempts):
                selector_hits[name] += 1

    if observed_queries == 0:
        status = '⚠️ NO OUTPUT'
    elif candidate_hits == expected_queries:
        status = '✅ FULL POOL'
    elif candidate_hits > 0:
        status = f'🟡 PARTIAL {candidate_hits}/{expected_queries}'
    else:
        status = '❌ UNSOLVED'

    def fmt_ranks(values):
        return '/'.join('-' if v is None else str(v) for v in values)

    summary_data.append({
        'id': puzzle_id,
        'status': status,
        'observed_queries': observed_queries,
        'expected_queries': expected_queries,
        'candidate_hits': candidate_hits,
        'selector_hits': selector_hits,
        'rank_probmul': fmt_ranks(rank_probmul),
        'rank_kgmon': fmt_ranks(rank_kgmon),
        'best_beam': f"{best_beam:.5f}" if best_beam is not None else 'N/A',
    })

print(
    f"{'Puzzle ID':<11} {'Status':<16} {'Pool':<7} {'CURRENT':<9} "
    f"{'IMB-EX':<9} {'QW-DIV':<9} {'RankA':<10} {'RankB':<10} {'BestBeam':<10}"
)
print("-" * 118)
for row in summary_data:
    denom = row['expected_queries']
    print(
        f"{row['id']:<11} {row['status']:<16} "
        f"{row['candidate_hits']}/{denom:<5} "
        f"{row['selector_hits']['CURRENT']}/{denom:<7} "
        f"{row['selector_hits']['IMBUE_EXACT']}/{denom:<7} "
        f"{row['selector_hits']['IMBUE_QWEN_DIVERSE']}/{denom:<7} "
        f"{row['rank_probmul']:<10} {row['rank_kgmon']:<10} {row['best_beam']:<10}"
    )

max_points = len(summary_data)
pool_points = sum(r['candidate_hits'] / r['expected_queries'] for r in summary_data)
selector_points = {
    name: sum(r['selector_hits'][name] / r['expected_queries'] for r in summary_data)
    for name in SELECTOR_NAMES
}

print("\n" + "=" * 118)
print(f"DEBUG weighted candidate-oracle points: {pool_points:.3f}/{max_points}")
for name in SELECTOR_NAMES:
    print(f"DEBUG weighted {name:24s}: {selector_points[name]:.3f}/{max_points}")
print("NOTE: selector hits above are computed from the ACTUAL two returned grids, fixing the old rank-1-only diagnostic bug.")
print("=" * 118)


# ----------------------------------------------------------------
# CHECK 5 — OUTPUT / FAILURE BREAKDOWN
# ----------------------------------------------------------------
print("\n" + "=" * 100)
print("CHECK 5: OUTPUT / FAILURE BREAKDOWN")
print("=" * 100)
no_output = [r['id'] for r in summary_data if r['observed_queries'] == 0]
partial_output = [
    r['id'] for r in summary_data
    if 0 < r['observed_queries'] < r['expected_queries']
]
no_correct_candidate = [
    r['id'] for r in summary_data
    if r['observed_queries'] > 0 and r['candidate_hits'] == 0
]
pool_has_correct = [r['id'] for r in summary_data if r['candidate_hits'] > 0]
print(f"No output at all:                    {len(no_output)} -> {no_output}")
print(f"Partial outputs (some queries miss): {len(partial_output)} -> {partial_output}")
print(f"Ran but no correct pool candidate:   {len(no_correct_candidate)} -> {no_correct_candidate}")
print(f"Pool has >=1 correct output:         {len(pool_has_correct)} -> {pool_has_correct}")


# ----------------------------------------------------------------
# CHECK 6 — MULTI-QUERY BREAKDOWN
# ----------------------------------------------------------------
print("\n" + "=" * 100)
print("CHECK 6: MULTI-QUERY PUZZLE BREAKDOWN")
print("=" * 100)
multi_query = [r for r in summary_data if r['expected_queries'] > 1]
if multi_query:
    print(f"Multi-query puzzles found: {len(multi_query)}")
    for r in multi_query:
        d = r['expected_queries']
        print(
            f"  {r['id']}: observed={r['observed_queries']}/{d}, pool={r['candidate_hits']}/{d}, "
            f"CURRENT={r['selector_hits']['CURRENT']}/{d}, "
            f"IMBUE_EXACT={r['selector_hits']['IMBUE_EXACT']}/{d}, "
            f"QWEN_DIVERSE={r['selector_hits']['IMBUE_QWEN_DIVERSE']}/{d}"
        )
else:
    print("No multi-query puzzles in DEBUG_PUZZLES.")


# ----------------------------------------------------------------
# CHECK 7 — TIME vs TIER CONSISTENCY (optional, manual log paste)
# ----------------------------------------------------------------
print("\n" + "=" * 100)
print("CHECK 7: TIME vs TIER CONSISTENCY")
print("=" * 100)
print("Paste the full run log text into `log_text` below to enable this check.")

log_text = """
PASTE_FULL_RUN_LOG_HERE
"""

times = {}
for match in re.finditer(r'finished (\w+) in ([\d.]+)s', log_text):
    times[match.group(1)] = float(match.group(2))

if times:
    print(f"Found timing data for {len(times)} puzzles:")
    for k, t in sorted(times.items(), key=lambda x: x[1]):
        if k in data_dict:
            tier, _ = get_puzzle_priority(k, data_dict)
            print(f"  {k}: {t:.1f}s, tier={tier}")
else:
    print("No timing data found — paste the run log into `log_text` to populate this check.")


# ----------------------------------------------------------------
# CHECK 8 — RARE COLOR PUZZLE BREAKDOWN
# ----------------------------------------------------------------
print("\n" + "=" * 100)
print("CHECK 8: RARE COLOR PUZZLE BREAKDOWN")
print("=" * 100)
RARE_COLOR_PUZZLES = [
    "6e4f6532", "195c6913", "62593bfd", "4e34c42c", "a32d8b75",
    "38007db0", "3a25b0d8", "5dbc8537", "d8e07eb2",
]
rare_color_results = [r for r in summary_data if r['id'] in RARE_COLOR_PUZZLES]
if rare_color_results:
    max_rare = len(rare_color_results)
    pool_rare = sum(r['candidate_hits'] / r['expected_queries'] for r in rare_color_results)
    print(f"Candidate-oracle weighted points: {pool_rare:.3f}/{max_rare}")
    for name in SELECTOR_NAMES:
        pts = sum(r['selector_hits'][name] / r['expected_queries'] for r in rare_color_results)
        print(f"{name:24s}: {pts:.3f}/{max_rare}")
    print("\nDetailed Rare Color Results:")
    for r in rare_color_results:
        d = r['expected_queries']
        print(
            f"  {r['id']}: {r['status']}, pool={r['candidate_hits']}/{d}, "
            f"CURRENT={r['selector_hits']['CURRENT']}/{d}, "
            f"IMB-EX={r['selector_hits']['IMBUE_EXACT']}/{d}, "
            f"QW-DIV={r['selector_hits']['IMBUE_QWEN_DIVERSE']}/{d}"
        )
else:
    print("No rare color puzzles found in DEBUG_PUZZLES.")


# ----------------------------------------------------------------
# CHECK 9 — HARDEST TIER 1 PUZZLE BREAKDOWN
# ----------------------------------------------------------------
print("\n" + "=" * 100)
print("CHECK 9: HARDEST TIER 1 PUZZLE BREAKDOWN")
print("=" * 100)
HARDEST_TIER1_PUZZLES = ["6ffbe589", "58f5dbd5", "5545f144", "291dc1e1"]
tier1_hard_results = [r for r in summary_data if r['id'] in HARDEST_TIER1_PUZZLES]
if tier1_hard_results:
    max_t1 = len(tier1_hard_results)
    pool_t1 = sum(r['candidate_hits'] / r['expected_queries'] for r in tier1_hard_results)
    print(f"Candidate-oracle weighted points: {pool_t1:.3f}/{max_t1}")
    for name in SELECTOR_NAMES:
        pts = sum(r['selector_hits'][name] / r['expected_queries'] for r in tier1_hard_results)
        print(f"{name:24s}: {pts:.3f}/{max_t1}")
    print("\nDetailed Hardest Tier 1 Results:")
    for r in tier1_hard_results:
        d = r['expected_queries']
        print(
            f"  {r['id']}: {r['status']}, pool={r['candidate_hits']}/{d}, "
            f"CURRENT={r['selector_hits']['CURRENT']}/{d}, "
            f"IMB-EX={r['selector_hits']['IMBUE_EXACT']}/{d}, "
            f"QW-DIV={r['selector_hits']['IMBUE_QWEN_DIVERSE']}/{d}"
        )
else:
    print("No hardest Tier 1 puzzles found in DEBUG_PUZZLES.")


# ----------------------------------------------------------------
# FINAL SUMMARY — TIER vs WEIGHTED COVERAGE/SELECTOR POINTS
# ----------------------------------------------------------------
print("\n" + "=" * 100)
print("FINAL SUMMARY: TIER vs WEIGHTED CANDIDATE / SELECTOR POINTS")
print("=" * 100)

for tier in [1, 2, 3]:
    rows = []
    for row in summary_data:
        if row['id'] in data_dict:
            t, _ = get_puzzle_priority(row['id'], data_dict)
            if t == tier:
                rows.append(row)
    if not rows:
        print(f"Tier {tier}: No puzzles in debug set")
        continue

    pool_pts = sum(r['candidate_hits'] / r['expected_queries'] for r in rows)
    selector_pts = {
        name: sum(r['selector_hits'][name] / r['expected_queries'] for r in rows)
        for name in SELECTOR_NAMES
    }
    print(
        f"Tier {tier}: pool={pool_pts:.3f}/{len(rows)}, "
        f"CURRENT={selector_pts['CURRENT']:.3f}, "
        f"IMBUE_EXACT={selector_pts['IMBUE_EXACT']:.3f}, "
        f"QWEN_DIVERSE={selector_pts['IMBUE_QWEN_DIVERSE']:.3f}"
    )

print("=" * 100)
print(f"ACTIVE PRODUCTION SELECTOR: {globals().get('PRODUCTION_SELECTOR', 'CURRENT')}")
print("DIAGNOSTIC COMPLETE.")
print("=" * 100)
